In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


class MLP:
    """
    Red Neuronal Multicapa (MLP) con Backpropagation.

    Parámetros
    ----------
    layer_sizes : list[int]
        Lista con el número de neuronas por capa, incluyendo entrada y salida.
        Ejemplo: [2, 4, 1] → 2 entradas, 1 capa oculta con 4 neuronas, 1 salida.
    learning_rate : float
        Tasa de aprendizaje para el descenso de gradiente.
    """

    def __init__(self, layer_sizes: list[int], learning_rate: float = 0.1):
        self.layer_sizes = layer_sizes
        self.lr = learning_rate
        self.n_layers = len(layer_sizes)
        self.loss_history: list[float] = []

        # Inicialización de pesos y biases (Xavier)
        np.random.seed(42)
        self.weights = []
        self.biases = []
        for i in range(self.n_layers - 1):
            fan_in = layer_sizes[i]
            fan_out = layer_sizes[i + 1]
            limit = np.sqrt(6.0 / (fan_in + fan_out))
            self.weights.append(np.random.uniform(-limit, limit, (fan_in, fan_out)))
            self.biases.append(np.zeros((1, fan_out)))

    # ------------------------------------------------------------------
    # Funciones de activación
    # ------------------------------------------------------------------

    @staticmethod
    def sigmoid(z: np.ndarray) -> np.ndarray:
        return 1.0 / (1.0 + np.exp(-z))

    @staticmethod
    def sigmoid_derivative(a: np.ndarray) -> np.ndarray:
        """Derivada de sigmoid expresada en términos de la activación a = sigmoid(z)."""
        return a * (1.0 - a)

    # ------------------------------------------------------------------
    # Forward pass
    # ------------------------------------------------------------------

    def forward(self, X: np.ndarray) -> np.ndarray:
        """
        Propaga X hacia adelante y almacena las activaciones de cada capa.

        Retorna la activación de la capa de salida.
        """
        self._activations = [X]
        a = X
        for W, b in zip(self.weights, self.biases):
            z = a @ W + b
            a = self.sigmoid(z)
            self._activations.append(a)
        return self._activations[-1]

    # ------------------------------------------------------------------
    # Backward pass (Backpropagation)
    # ------------------------------------------------------------------

    def backward(self, y: np.ndarray) -> None:
        """
        Calcula los gradientes mediante retropropagación y actualiza pesos y biases.
        Las activaciones se toman de self._activations, guardadas en el forward pass.

        Parámetros
        ----------
        y : ndarray — etiquetas verdaderas
        """
        m = y.shape[0]                     # número de muestras
        activations = self._activations

        # Error en la capa de salida: dL/da * da/dz
        delta = (activations[-1] - y) * self.sigmoid_derivative(activations[-1])

        # Propagar el error capa por capa hacia atrás
        for i in reversed(range(len(self.weights))):
            grad_W = activations[i].T @ delta / m
            grad_b = np.sum(delta, axis=0, keepdims=True) / m

            # Delta para la capa anterior (si existe)
            if i > 0:
                delta = (delta @ self.weights[i].T) * self.sigmoid_derivative(activations[i])

            # Actualización de parámetros
            self.weights[i] -= self.lr * grad_W
            self.biases[i]  -= self.lr * grad_b

    # ------------------------------------------------------------------
    # Entrenamiento
    # ------------------------------------------------------------------

    def train(self, X: np.ndarray, y: np.ndarray, epochs: int = 1000) -> None:
        """Entrena la red durante `epochs` épocas."""
        self.loss_history = []
        for epoch in range(1, epochs + 1):
            y_pred = self.forward(X)
            self.backward(y)

            # Error cuadrático medio (MSE)
            loss = float(np.mean((y_pred - y) ** 2))
            self.loss_history.append(loss)

            if epoch % 500 == 0 or epoch == 1:
                print(f"Época {epoch:>6} | Pérdida (MSE): {loss:.6f}")

    # ------------------------------------------------------------------
    # Predicción
    # ------------------------------------------------------------------

    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        """Retorna etiquetas binarias usando el umbral dado."""
        return (self.forward(X) >= threshold).astype(int)


# ======================================================================
# Demo: problema XOR
# ======================================================================

def main():
    # Datos XOR
    X = np.array([[0, 0],
                  [0, 1],
                  [1, 0],
                  [1, 1]], dtype=float)
    y = np.array([[0],
                  [1],
                  [1],
                  [0]], dtype=float)

    # Configuración de hiperparámetros
    layer_sizes   = [2, 4, 1]   # [entrada, neuronas ocultas, salida]
    learning_rate = 0.5
    epochs        = 10_000

    print("=" * 50)
    print(f"Arquitectura : {layer_sizes}")
    print(f"Learning rate: {learning_rate}")
    print(f"Épocas       : {epochs}")
    print("=" * 50)

    # Crear y entrenar la red
    mlp = MLP(layer_sizes=layer_sizes, learning_rate=learning_rate)
    mlp.train(X, y, epochs=epochs)

    # Resultados
    y_pred_proba = mlp.forward(X)
    y_pred_bin   = mlp.predict(X)

    df = pd.DataFrame({
        "x1"              : X[:, 0].astype(int),
        "x2"              : X[:, 1].astype(int),
        "XOR esperado"    : y.flatten().astype(int),
        "Probabilidad"    : y_pred_proba.flatten().round(4),
        "Predicción (bin)": y_pred_bin.flatten(),
    })

    print("\nResultados XOR:")
    print(df.to_string(index=False))

    accuracy = (df["XOR esperado"] == df["Predicción (bin)"]).mean() * 100
    print(f"\nExactitud: {accuracy:.1f}%")

    # Curva de pérdida
    plt.figure(figsize=(8, 4))
    plt.plot(mlp.loss_history, color="steelblue", linewidth=1.5)
    plt.title("Curva de pérdida — MLP XOR")
    plt.xlabel("Época")
    plt.ylabel("MSE")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    main()
